# 03 — Binarização dos patches (versão enxuta: só Thermometer-HSV)

**Objetivo**: converter os patches RGB salvos no notebook 02 em vetores binários Thermometer-HSV, prontos para a WiSARD/ClusWiSARD.

## O que mudou em relação ao 03 original

O notebook 03 original implementava e processava **5 esquemas** (otsu, adaptive, canny, thermometer-rgb, thermometer-hsv) × 3 splits = 15 arquivos, ~900 MB, dezenas de minutos. A fase experimental já concluiu que **thermometer-HSV é o melhor** (F1=0,5981 no WiSARD; Otsu provou ser fraco; os demais foram descartados).

Esta versão gera **apenas o thermometer-HSV** (3 arquivos), cortando o tempo para ~1/5 e o disco para ~500 MB. **A função de binarização é idêntica à original** (mesma conversão HSV, mesmos thresholds `k/(levels+1)`, `levels=4`), então os vetores são bit-a-bit iguais aos que produziram os resultados históricos — a comparabilidade está preservada.

As visualizações comparativas (5 esquemas lado a lado, histogramas de densidade) foram removidas: o seu colega já as validou e a análise está documentada. Se quiser revisá-las, use o 03 original.

## 0. Setup

`PATCHES_DIR` deve apontar para onde o **notebook 02** salvou os `.npz` (por padrão `./patches`). Rode os dois notebooks no mesmo diretório de trabalho e isto já estará certo.

In [1]:
from pathlib import Path
import numpy as np
import cv2
from tqdm import tqdm

PATCHES_DIR = Path("./patches")     # onde o notebook 02 salvou patches_*_48.npz
BIN_DIR     = Path("./binarized")   # saída desta etapa
BIN_DIR.mkdir(exist_ok=True)

PATCH_SIZE = 32
SPLITS = ["train", "valid", "test"]

# Hiperparâmetro do thermometer (idêntico ao 03 original)
THERMOMETER_LEVELS = 4

EXPECTED_BITS = PATCH_SIZE * PATCH_SIZE * 3 * THERMOMETER_LEVELS  # 48*48*3*4 = 27648

# Sanidade dos insumos
missing = [s for s in SPLITS
           if not (PATCHES_DIR / f"patches_{s}_{PATCH_SIZE}.npz").exists()]
if missing:
    raise FileNotFoundError(
        f"Faltam patches dos splits {missing} em {PATCHES_DIR}. "
        f"Rode o notebook 02 antes."
    )

print("Setup OK.")
print(f"  Patch {PATCH_SIZE}x{PATCH_SIZE}, thermometer levels={THERMOMETER_LEVELS}")
print(f"  Vetor binário esperado por patch: {EXPECTED_BITS} bits")
print(f"  Insumos encontrados em {PATCHES_DIR}: OK")

Setup OK.
  Patch 24x24, thermometer levels=4
  Vetor binário esperado por patch: 6912 bits
  Insumos encontrados em patches: OK


## 1. Função de binarização Thermometer-HSV

**Cópia exata** da implementação do notebook 03 original (caminho `color_space="hsv"`). Cada pixel, em cada um dos 3 canais HSV, vira `levels` bits no estilo "termômetro": os primeiros K bits ligados, proporcional à intensidade, com thresholds em `k/(levels+1)`.

In [2]:
def binarize_thermometer_hsv(patch_bgr, levels=THERMOMETER_LEVELS):
    """Thermometer encoding no espaço HSV. Idêntico ao 03 original.

    Exemplo (levels=4, intensidade 0.6): thresholds=[0.2,0.4,0.6,0.8]
    -> bits=[1,1,1,0]. Saída: vetor (H*W*3*levels,) uint8.
    """
    img = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    img = img.astype(np.float32) / 255.0

    bits_list = []
    for c in range(3):  # 3 canais (H, S, V)
        channel = img[:, :, c]
        for k in range(1, levels + 1):
            thr = k / (levels + 1)
            bits_list.append((channel >= thr).astype(np.uint8).flatten())
    return np.concatenate(bits_list)


# Verificação rápida do tamanho do vetor
_dummy = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), np.uint8)
_v = binarize_thermometer_hsv(_dummy)
assert len(_v) == EXPECTED_BITS, f"esperado {EXPECTED_BITS}, obtido {len(_v)}"
print(f"binarize_thermometer_hsv OK — vetor de {len(_v)} bits por patch")

binarize_thermometer_hsv OK — vetor de 6912 bits por patch


## 2. Processar todos os patches e salvar

Para cada split: carrega os patches RGB do notebook 02, binariza, empacota com `np.packbits` (8 bits/byte; `bits_per_patch` guardado para desempacotar depois) e salva em `bin_thermometer_hsv_{split}_48.npz`.

In [3]:
def binarize_all_patches(X, binarize_fn, desc=""):
    """Aplica binarize_fn a todos os patches. Retorna array (N, bits) uint8."""
    sample = binarize_fn(X[0])
    bits_per_patch = len(sample)
    out = np.zeros((len(X), bits_per_patch), dtype=np.uint8)
    out[0] = sample
    for i in tqdm(range(1, len(X)), desc=desc):
        out[i] = binarize_fn(X[i])
    return out


SCHEME_NAME = "thermometer_hsv"

for split in SPLITS:
    print(f"\n=== {SCHEME_NAME} / {split} ===")
    data = np.load(PATCHES_DIR / f"patches_{split}_{PATCH_SIZE}.npz")
    X, y = data["X"], data["y"]
    print(f"  patches carregados: {X.shape}")

    X_bin = binarize_all_patches(X, binarize_thermometer_hsv,
                                 desc=f"{SCHEME_NAME}/{split}")
    bits_per_patch = X_bin.shape[1]
    X_packed = np.packbits(X_bin, axis=1)

    out_path = BIN_DIR / f"bin_{SCHEME_NAME}_{split}_{PATCH_SIZE}.npz"
    np.savez_compressed(out_path, X_packed=X_packed, y=y,
                        bits_per_patch=bits_per_patch)
    size_mb = out_path.stat().st_size / 1e6
    print(f"  salvo: {out_path.name}  "
          f"({len(X_bin)} patches × {bits_per_patch} bits → {size_mb:.1f} MB)")


=== thermometer_hsv / train ===
  patches carregados: (296685, 24, 24, 3)


thermometer_hsv/train: 100%|██████████| 296684/296684 [00:40<00:00, 7314.46it/s]


  salvo: bin_thermometer_hsv_train_24.npz  (296685 patches × 6912 bits → 67.6 MB)

=== thermometer_hsv / valid ===
  patches carregados: (226382, 24, 24, 3)


thermometer_hsv/valid: 100%|██████████| 226381/226381 [00:31<00:00, 7151.75it/s]


  salvo: bin_thermometer_hsv_valid_24.npz  (226382 patches × 6912 bits → 56.1 MB)

=== thermometer_hsv / test ===
  patches carregados: (115333, 24, 24, 3)


thermometer_hsv/test: 100%|██████████| 115332/115332 [00:15<00:00, 7657.45it/s]


  salvo: bin_thermometer_hsv_test_24.npz  (115333 patches × 6912 bits → 28.4 MB)


## 3. Utilitário `load_binarized` + sanidade

Mesma função usada pelos notebooks 04*. Recarrega os arquivos salvos, desempacota, confere shapes e contagens. É exatamente isto que o notebook 04d vai consumir.

In [4]:
def load_binarized(scheme_name, split, patch_size=PATCH_SIZE, bin_dir=BIN_DIR):
    path = bin_dir / f"bin_{scheme_name}_{split}_{patch_size}.npz"
    data = np.load(path)
    X_packed = data["X_packed"]
    y = data["y"]
    bits_per_patch = int(data["bits_per_patch"])
    X = np.unpackbits(X_packed, axis=1)[:, :bits_per_patch]
    return X, y


print(f"{'split':<6} {'X shape':<22} {'bits':>7} {'pos':>7} {'neg':>7}")
print("-" * 56)
for split in SPLITS:
    X, y = load_binarized(SCHEME_NAME, split)
    n_pos = int((y == "pos").sum())
    n_neg = int((y == "neg").sum())
    assert X.shape[1] == EXPECTED_BITS, \
        f"{split}: bits inesperados {X.shape[1]} (esperado {EXPECTED_BITS})"
    assert set(np.unique(X)).issubset({0, 1}), f"{split}: X não é binário"
    print(f"{split:<6} {str(X.shape):<22} {X.shape[1]:>7} {n_pos:>7} {n_neg:>7}")

print("\nBinarização concluída. Próximo: notebook 04d (varredura ClusWiSARD).")

split  X shape                   bits     pos     neg
--------------------------------------------------------
train  (296685, 6912)            6912   98895  197790
valid  (226382, 6912)            6912    7007  219375
test   (115333, 6912)            6912    3402  111931

Binarização concluída. Próximo: notebook 04d (varredura ClusWiSARD).


## 4. Notas

- `bits_per_patch = 27648 = 48×48×3×4`. Esse é o tamanho que o notebook 04d usa para conferir divisibilidade por `addressSize` (27648/32=864, 27648/48=576 — ambos exatos).
- O `np.packbits` reduz o disco ~8× sem perda: 27648 bits → 3456 bytes por patch antes da compressão `.npz`.
- Se algum dia precisar dos outros esquemas (Otsu etc.) para uma seção de ablacão no relatório, use o notebook 03 original — a binarização HSV aqui é a mesma, então os resultados continuam comparáveis.